# Machine learning foundations
Completed notebook for the Shortcut Press guide. Run the cells from top to bottom; results match the guide because every split uses `random_state=42`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

## Lessons 2–3: house prices

In [ ]:
houses = pd.read_csv("houses.csv")
houses.describe()

In [ ]:
houses.select_dtypes("number").corr()["price_k"].round(2)

In [ ]:
num = ["size_m2", "bedrooms", "age_years", "distance_km"]
X = houses[num]
y = houses["price_k"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2), " R2:", round(r2_score(y_test, pred), 3))
dict(zip(num, model.coef_.round(2)))

## Lesson 4: churn

In [ ]:
churn = pd.read_csv("churn.csv")
Xc = churn.drop(columns="churned")
yc = churn["churned"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)
num_c = ["tenure_months", "monthly_fee", "support_calls"]
log = LogisticRegression(max_iter=1000).fit(Xc_train[num_c], yc_train)
pc = log.predict(Xc_test[num_c])
print(accuracy_score(yc_test, pc))
confusion_matrix(yc_test, pc)

## Lesson 5: a pipeline

In [ ]:
prep = ColumnTransformer([
    ("num", StandardScaler(), num_c),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["plan", "auto_pay"]),
])
pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
pipe.fit(Xc_train, yc_train)
pc2 = pipe.predict(Xc_test)
print(accuracy_score(yc_test, pc2), precision_score(yc_test, pc2), recall_score(yc_test, pc2))

## Lesson 6: trees and forests

In [ ]:
Xd = pd.get_dummies(Xc, columns=["plan", "auto_pay"])
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yc, test_size=0.25, random_state=42, stratify=yc)
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xd_train, yd_train)
print("depth 3:", tree.score(Xd_train, yd_train), tree.score(Xd_test, yd_test))
forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xd_train, yd_train)
print("forest:", forest.score(Xd_test, yd_test))
pd.Series(forest.feature_importances_, index=Xd.columns).sort_values(ascending=False).round(3)

## Lesson 7: validation and tuning

In [ ]:
scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), Xd_train, yd_train, cv=5)
print(scores.round(3), scores.mean().round(3))
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), {"max_depth": [2, 3, 4, 5, 6, None]}, cv=5).fit(Xd_train, yd_train)
print(grid.best_params_, round(grid.best_score_, 3), grid.score(Xd_test, yd_test))

## Lesson 8: save and use the model

In [ ]:
joblib.dump(pipe, "churn_model.joblib")
loaded = joblib.load("churn_model.joblib")
new = pd.DataFrame([[3, 55.0, 4, "Pro", "no"], [48, 15.0, 0, "Basic", "yes"]], columns=Xc.columns)
loaded.predict_proba(new)[:, 1].round(3)